# 📝 Reusable Prompt Templates

**Build a library of production-ready prompt templates**

---

## 📋 Overview

**What you'll learn:**
- Template design patterns
- Variable substitution
- Template library architecture
- LangChain PromptTemplate
- Production best practices

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
from typing import Dict, List, Optional
import os
import json

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🎯 Why Templates?

**Without templates:**
```python
# Hardcoded, not reusable
prompt = "Summarize this article: ..."  # ❌
```

**With templates:**
```python
# Reusable, testable, versioned
template = PromptTemplate(
    "Summarize {content} in {max_words} words"
)
prompt = template.format(content=article, max_words=50)  # ✅
```

**Benefits:**
- ♻️ **Reusability**: Use across projects
- 🧪 **Testing**: Easy to test variations
- 📦 **Versioning**: Track changes
- 🎯 **Consistency**: Same format everywhere
- 🔄 **Iteration**: Improve over time

## 🏗️ Basic Template Class

In [ ]:
class PromptTemplate:
    """Simple prompt template with variable substitution."""
    
    def __init__(self, template: str, name: str = None):
        """
        Args:
            template: Template string with {variables}
            name: Optional name for the template
        """
        self.template = template
        self.name = name or "unnamed"
    
    def format(self, **kwargs) -> str:
        """Format template with variables."""
        try:
            return self.template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing variable in template: {e}")
    
    def get_variables(self) -> List[str]:
        """Extract variable names from template."""
        import re
        return re.findall(r'\{(\w+)\}', self.template)
    
    def __repr__(self):
        return f"PromptTemplate(name='{self.name}', vars={self.get_variables()})"

# Test it
template = PromptTemplate(
    "Translate '{text}' to {language}",
    name="translator"
)

print(template)
print(f"Variables: {template.get_variables()}")
print(f"\nExample:\n{template.format(text='Hello', language='Spanish')}")

## 📚 Template Library

In [ ]:
class TemplateLibrary:
    """Collection of reusable prompt templates."""
    
    # Summarization
    SUMMARIZE = PromptTemplate(
        """Summarize the following text in {max_words} words or less.
Focus on the main points and key takeaways.

Text: {text}

Summary:""",
        name="summarize"
    )
    
    # Translation
    TRANSLATE = PromptTemplate(
        """Translate the following text from {source_lang} to {target_lang}.
Maintain the tone and style of the original.

Text: {text}

Translation:""",
        name="translate"
    )
    
    # Sentiment Analysis
    SENTIMENT = PromptTemplate(
        """Analyze the sentiment of this text.
Respond with ONLY one word: Positive, Negative, or Neutral.

Text: {text}

Sentiment:""",
        name="sentiment"
    )
    
    # Code Explanation
    CODE_EXPLAIN = PromptTemplate(
        """Explain this {language} code in simple terms.
Include what it does and how it works.

Code:
```{language}
{code}
```

Explanation:""",
        name="code_explain"
    )
    
    # Question Answering
    QA = PromptTemplate(
        """Answer the question based on the context below.
If the answer is not in the context, say "I don't have enough information."

Context: {context}

Question: {question}

Answer:""",
        name="qa"
    )
    
    # Data Extraction
    EXTRACT = PromptTemplate(
        """Extract {fields} from the text below.
Return as JSON.

Text: {text}

JSON:""",
        name="extract"
    )
    
    # Classification
    CLASSIFY = PromptTemplate(
        """Classify this text into one of these categories: {categories}
Respond with ONLY the category name.

Text: {text}

Category:""",
        name="classify"
    )

# Test templates
print("📚 Available Templates:\n")
for attr in dir(TemplateLibrary):
    if not attr.startswith('_'):
        template = getattr(TemplateLibrary, attr)
        if isinstance(template, PromptTemplate):
            print(f"  {template.name}: {template.get_variables()}")

In [ ]:
# Use templates in practice
def test_template(template: PromptTemplate, **kwargs) -> str:
    """Test a template with real LLM."""
    prompt = template.format(**kwargs)
    print(f"🔍 Testing: {template.name}")
    print(f"Prompt:\n{prompt}\n")
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100
    )
    
    result = response.choices[0].message.content
    print(f"Result: {result}\n")
    return result

# Test sentiment analysis
test_template(
    TemplateLibrary.SENTIMENT,
    text="I absolutely love this product! It's amazing!"
)

# Test classification
test_template(
    TemplateLibrary.CLASSIFY,
    text="Python is a great programming language for data science.",
    categories="Technology, Sports, Politics, Entertainment"
)

## 🎨 Advanced Templates with Examples

In [ ]:
class FewShotTemplate:
    """Template with few-shot examples."""
    
    def __init__(self, instruction: str, examples: List[Dict], name: str = None):
        """
        Args:
            instruction: Task instruction
            examples: List of {'input': ..., 'output': ...}
            name: Template name
        """
        self.instruction = instruction
        self.examples = examples
        self.name = name or "few_shot"
    
    def format(self, input_text: str) -> str:
        """Format template with examples."""
        prompt_parts = [self.instruction, ""]
        
        # Add examples
        for i, example in enumerate(self.examples, 1):
            prompt_parts.append(f"Example {i}:")
            prompt_parts.append(f"Input: {example['input']}")
            prompt_parts.append(f"Output: {example['output']}")
            prompt_parts.append("")
        
        # Add actual input
        prompt_parts.append("Now you try:")
        prompt_parts.append(f"Input: {input_text}")
        prompt_parts.append("Output:")
        
        return "\n".join(prompt_parts)

# Example: Sentiment classification with few-shot
sentiment_template = FewShotTemplate(
    instruction="Classify the sentiment as Positive, Negative, or Neutral.",
    examples=[
        {"input": "I love this!", "output": "Positive"},
        {"input": "This is terrible.", "output": "Negative"},
        {"input": "It's okay.", "output": "Neutral"},
    ],
    name="sentiment_few_shot"
)

# Test it
prompt = sentiment_template.format("The product works as expected.")
print("Few-Shot Prompt:")
print(prompt)
print("\n" + "="*50 + "\n")

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=10
)

print(f"Result: {response.choices[0].message.content}")

## 🔧 Chain-of-Thought Templates

In [ ]:
class ChainOfThoughtTemplate:
    """Template that encourages step-by-step reasoning."""
    
    MATH_PROBLEM = PromptTemplate(
        """Solve this math problem step by step.

Problem: {problem}

Let's solve this step by step:
1.""",
        name="math_cot"
    )
    
    REASONING = PromptTemplate(
        """Question: {question}

Let's think through this step by step:
1) First, let's identify what we know:
2) Next, let's consider:
3) Therefore:

Answer:""",
        name="reasoning_cot"
    )
    
    DEBUG = PromptTemplate(
        """Debug this {language} code by thinking step by step.

Code:
```{language}
{code}
```

Error: {error}

Let's debug this step by step:
1) What is the code trying to do?
2) What is the error telling us?
3) Where is the bug?
4) How to fix it?

Solution:""",
        name="debug_cot"
    )

# Test CoT template
prompt = ChainOfThoughtTemplate.MATH_PROBLEM.format(
    problem="If a train travels 60 miles in 45 minutes, what is its speed in miles per hour?"
)

print("Chain-of-Thought Prompt:")
print(prompt)
print("\n" + "="*50 + "\n")

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200
)

print(f"Result:\n{response.choices[0].message.content}")

## 💾 Save and Load Templates

In [ ]:
import json
from pathlib import Path

class TemplateManager:
    """Manage and persist prompt templates."""
    
    def __init__(self, templates_dir: str = "templates"):
        self.templates_dir = Path(templates_dir)
        self.templates_dir.mkdir(exist_ok=True)
        self.templates: Dict[str, PromptTemplate] = {}
    
    def save_template(self, template: PromptTemplate):
        """Save template to disk."""
        template_data = {
            'name': template.name,
            'template': template.template,
            'variables': template.get_variables()
        }
        
        file_path = self.templates_dir / f"{template.name}.json"
        with open(file_path, 'w') as f:
            json.dump(template_data, f, indent=2)
        
        print(f"✅ Saved template: {template.name}")
    
    def load_template(self, name: str) -> PromptTemplate:
        """Load template from disk."""
        file_path = self.templates_dir / f"{name}.json"
        
        if not file_path.exists():
            raise FileNotFoundError(f"Template not found: {name}")
        
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        return PromptTemplate(data['template'], name=data['name'])
    
    def list_templates(self) -> List[str]:
        """List all saved templates."""
        return [f.stem for f in self.templates_dir.glob('*.json')]
    
    def register(self, template: PromptTemplate):
        """Register template in memory."""
        self.templates[template.name] = template
    
    def get(self, name: str) -> PromptTemplate:
        """Get template from memory or load from disk."""
        if name in self.templates:
            return self.templates[name]
        
        template = self.load_template(name)
        self.templates[name] = template
        return template

# Test template manager
manager = TemplateManager()

# Save templates
manager.save_template(TemplateLibrary.SUMMARIZE)
manager.save_template(TemplateLibrary.SENTIMENT)
manager.save_template(TemplateLibrary.TRANSLATE)

# List templates
print(f"\n📚 Saved templates: {manager.list_templates()}")

# Load and use
template = manager.load_template('sentiment')
prompt = template.format(text="This is amazing!")
print(f"\nLoaded template:\n{prompt}")

## 🎯 Production Template System

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class TemplateConfig:
    """Configuration for template execution."""
    model: str = "gpt-3.5-turbo"
    temperature: float = 0.7
    max_tokens: int = 500
    stop_sequences: List[str] = None

class ProductionTemplateEngine:
    """Production-ready template engine."""
    
    def __init__(self, client: OpenAI):
        self.client = client
        self.manager = TemplateManager()
        self._register_default_templates()
    
    def _register_default_templates(self):
        """Register default templates."""
        for attr in dir(TemplateLibrary):
            if not attr.startswith('_'):
                template = getattr(TemplateLibrary, attr)
                if isinstance(template, PromptTemplate):
                    self.manager.register(template)
    
    def execute(
        self,
        template_name: str,
        variables: Dict[str, Any],
        config: TemplateConfig = None
    ) -> Dict[str, Any]:
        """Execute template with LLM."""
        config = config or TemplateConfig()
        
        # Get template
        template = self.manager.get(template_name)
        
        # Format prompt
        prompt = template.format(**variables)
        
        # Call LLM
        response = self.client.chat.completions.create(
            model=config.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=config.temperature,
            max_tokens=config.max_tokens,
            stop=config.stop_sequences
        )
        
        return {
            'template': template_name,
            'prompt': prompt,
            'result': response.choices[0].message.content,
            'tokens': response.usage.total_tokens,
            'model': config.model
        }
    
    def batch_execute(
        self,
        template_name: str,
        batch_variables: List[Dict[str, Any]],
        config: TemplateConfig = None
    ) -> List[Dict[str, Any]]:
        """Execute template for multiple inputs."""
        results = []
        for variables in batch_variables:
            result = self.execute(template_name, variables, config)
            results.append(result)
        return results

# Test production engine
engine = ProductionTemplateEngine(client)

# Single execution
result = engine.execute(
    'sentiment',
    {'text': 'This product exceeded my expectations!'},
    config=TemplateConfig(temperature=0, max_tokens=10)
)

print("📊 Execution Result:")
print(f"  Template: {result['template']}")
print(f"  Result: {result['result']}")
print(f"  Tokens: {result['tokens']}")

# Batch execution
print("\n📦 Batch Execution:")
batch_results = engine.batch_execute(
    'sentiment',
    [
        {'text': 'I love it!'},
        {'text': 'Not great.'},
        {'text': 'It\'s fine.'},
    ],
    config=TemplateConfig(temperature=0, max_tokens=10)
)

for i, result in enumerate(batch_results, 1):
    print(f"  {i}. {result['result']}")

## ✅ Summary

### Key Concepts:

1. **📝 Basic Templates**
   - Variable substitution with `{variable}`
   - Reusable across projects
   
2. **📚 Template Libraries**
   - Organize by use case
   - Summarization, translation, classification, etc.
   
3. **🎨 Advanced Templates**
   - Few-shot with examples
   - Chain-of-thought for reasoning
   
4. **💾 Template Management**
   - Save/load from disk
   - Version control
   - Share across team

### Template Design Best Practices:

```python
# ✅ Good: Clear, specific, with examples
template = """
Task: {task}

Examples:
{examples}

Input: {input}
Output:
"""

# ❌ Bad: Vague, no structure
template = "Do {task} for {input}"
```

### Production Checklist:

- ✅ **Version control**: Track template changes
- ✅ **Test variations**: A/B test templates
- ✅ **Document**: Explain when to use each
- ✅ **Validate inputs**: Check required variables
- ✅ **Error handling**: Graceful failures
- ✅ **Logging**: Track template usage
- ✅ **Metrics**: Measure success rates

### Template Organization:

```
templates/
  ├── summarization/
  │   ├── short.json
  │   ├── detailed.json
  │   └── bullet_points.json
  ├── classification/
  │   ├── sentiment.json
  │   └── topic.json
  └── extraction/
      ├── entities.json
      └── key_points.json
```

### Common Template Patterns:

1. **Instruction + Context + Question**
   ```
   You are {role}.
   
   Context: {context}
   
   Question: {question}
   ```

2. **Few-Shot Learning**
   ```
   Task: {task}
   
   Example 1: {example1}
   Example 2: {example2}
   
   Now you try: {input}
   ```

3. **Chain-of-Thought**
   ```
   Problem: {problem}
   
   Let's solve step by step:
   1.
   ```

### Next: `03_prompt_engineering/05_chain_of_thought.ipynb`